# Training BISINDO A-Z - CPU, Rhio + Sanjaya

Upload **file notebook ini saja** ke Google Colab dan jalankan sel dari atas ke bawah. Gunakan runtime CPU; GPU tidak diperlukan. Notebook mengunduh dua dataset, mengekstrak landmark dengan dua worker browser, menyimpan checkpoint setiap foto, lalu melatih Random Forest di CPU.

Jika ekstraksi berhenti, jalankan kembali sel ekstraksi. Output awal akan menampilkan jumlah checkpoint dan hanya memproses foto yang belum selesai. Checkpoint menggunakan NDJSON append, sehingga hasil lama tidak ditimpa. Satu baris terakhir yang terpotong akibat runtime mati akan diabaikan dengan aman.

Sumber: Rhio Sutoyo dkk. (MIT) dan Samuel Ady Sanjaya, 18 Oktober 2024, v1, DOI 10.17632/ywnjpbcz8m.1 (CC BY 4.0). Hanya Original Images Sanjaya yang digunakan. Duplikat berlabel konflik dikeluarkan dan dicatat. Tidak ada horizontal flip atau augmentation.

Ekstraksi memakai MediaPipe Tasks Vision 1.0.1, model float16 v1, canonicalization handedness dan extractor 52 fitur yang sama dengan website. Untuk mempercepat dataset foto statis, landmarker menggunakan IMAGE mode pada kanvas 640x480. Ini memakai provider/model yang sama, tetapi bukan pengujian runtime VIDEO kamera. Model A-Z tetap kandidat eksperimen sampai parity browser dan pengujian kamera selesai. Foto statis juga tidak membuktikan gesture dinamis.


In [ ]:
import json, pathlib, subprocess, os, shutil, urllib.request, hashlib
ROOT = pathlib.Path('/content/bisindo-alphabet-training')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
def run_live(*args):
    process = subprocess.Popen(args, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'Perintah gagal dengan exit code {code}: {args}')
print('Python:', os.sys.version.split()[0])
run_live('node', '--version')


In [ ]:
# Menulis seluruh source yang tertanam di notebook. Tidak perlu upload project.
bundle = json.loads("{\"src/lib/config/tracking.ts\":\"// Engineering tracking thresholds, not BISINDO correctness thresholds.\\nexport const trackingConfig = {\\n  packageVersion: \\\"1.0.1\\\",\\n  assetId: \\\"mediapipe-hand-landmarker-float16-v1\\\",\\n  modelPath: \\\"/models/mediapipe/hand_landmarker-float16-v1.task\\\",\\n  wasmRoot: \\\"/models/mediapipe/tasks-vision-1.0.1/wasm\\\",\\n  minHandednessScore: 0.7,\\n  minCategoryMargin: 0.2,\\n  minIntervalMs: 80,\\n  statusIntervalMs: 250,\\n} as const;\\n\",\"src/lib/mediapipe/canonicalize.ts\":\"import type { HandLandmarkerResult } from \\\"@mediapipe/tasks-vision\\\";\\nimport type { CanonicalResult, HandFrame, Landmark, HandRequirementStatus } from \\\"@/types/tracking\\\";\\nimport type { SignContent } from \\\"@/types/content\\\";\\nimport { trackingConfig } from \\\"@/lib/config/tracking\\\";\\n\\ntype RawHands = Pick<HandLandmarkerResult, \\\"landmarks\\\" | \\\"worldLandmarks\\\" | \\\"handedness\\\">;\\nconst validLandmarks = (points: Landmark[]) => points.length === 21 && points.every((point) => [point.x, point.y, point.z].every(Number.isFinite));\\n\\nexport function canonicalizeHands(raw: RawHands, timestampMs: number): CanonicalResult {\\n  const frame: HandFrame = { timestampMs, left: null, right: null };\\n  const uncertain = (): CanonicalResult => ({ frame: { timestampMs, left: null, right: null }, ambiguous: true });\\n  if (!Number.isFinite(timestampMs) || raw.landmarks.length > 2 || raw.handedness.length !== raw.landmarks.length) return uncertain();\\n  for (let index = 0; index < raw.landmarks.length; index++) {\\n    const points = raw.landmarks[index];\\n    const categories = [...(raw.handedness[index] ?? [])].sort((a, b) => b.score - a.score);\\n    const category = categories[0];\\n    if (!points || !validLandmarks(points) || !category || !Number.isFinite(category.score) || category.score < trackingConfig.minHandednessScore || category.score > 1) return uncertain();\\n    if (categories[1] && category.score - categories[1].score < trackingConfig.minCategoryMargin) return uncertain();\\n    // Index only pairs fields of the same detection. The provider label selects the slot.\\n    const slot = category.categoryName === \\\"Left\\\" ? \\\"left\\\" : category.categoryName === \\\"Right\\\" ? \\\"right\\\" : null;\\n    if (!slot || frame[slot]) return uncertain();\\n    const world = raw.worldLandmarks[index];\\n    if (world && !validLandmarks(world)) return uncertain();\\n    frame[slot] = {\\n      side: slot === \\\"left\\\" ? \\\"LEFT\\\" : \\\"RIGHT\\\",\\n      handednessScore: category.score,\\n      landmarks: points.map(({ x, y, z }) => ({ x, y, z })),\\n      ...(world ? { worldLandmarks: world.map(({ x, y, z }) => ({ x, y, z })) } : {}),\\n    };\\n  }\\n  return { frame, ambiguous: false };\\n}\\n\\nexport function checkRequiredHands(result: CanonicalResult, sign: Pick<SignContent, \\\"requiredHands\\\" | \\\"handednessPolicy\\\">): HandRequirementStatus {\\n  if (result.ambiguous) return \\\"UNCERTAIN\\\";\\n  const { left, right } = result.frame;\\n  const count = Number(!!left) + Number(!!right);\\n  if (!count) return \\\"NO_HAND\\\";\\n  if (sign.requiredHands === \\\"TWO\\\" && count < 2) return \\\"INSUFFICIENT_HANDS\\\";\\n  if (sign.requiredHands === \\\"ONE\\\" && count > 1) return \\\"UNCERTAIN\\\";\\n  if (sign.handednessPolicy === \\\"LEFT\\\" && !left || sign.handednessPolicy === \\\"RIGHT\\\" && !right || sign.handednessPolicy === \\\"VALIDATOR_DEFINED\\\") return \\\"UNCERTAIN\\\";\\n  // UNSPECIFIED allows tracking, never a claim that either hand is linguistically correct.\\n  return \\\"TRACKING\\\";\\n}\\n\",\"src/features/recognition/features.ts\":\"import type { CanonicalHand, HandFrame, Landmark } from \\\"@/types/tracking\\\";\\nimport { trackingConfig } from \\\"@/lib/config/tracking\\\";\\n\\nexport const featureSchema = {\\n  id: \\\"hands-geometry-v1\\\",\\n  normalizationVersion: \\\"world-palm-scale-v1\\\",\\n  landmarkerAssetId: trackingConfig.assetId,\\n  length: 52,\\n  handLength: 25,\\n} as const;\\n\\nconst joints = [[1, 2, 3], [2, 3, 4], [5, 6, 7], [6, 7, 8], [9, 10, 11], [10, 11, 12], [13, 14, 15], [14, 15, 16], [17, 18, 19], [18, 19, 20]] as const;\\nconst tips = [4, 8, 12, 16, 20] as const;\\nconst subtract = (a: Landmark, b: Landmark): Landmark => ({ x: a.x - b.x, y: a.y - b.y, z: a.z - b.z });\\nconst norm = (a: Landmark) => Math.hypot(a.x, a.y, a.z);\\nconst distance = (a: Landmark, b: Landmark) => norm(subtract(a, b));\\nconst valid = (points: Landmark[]) => points.length === 21 && points.every((point) => [point.x, point.y, point.z].every(Number.isFinite));\\n\\nfunction extractHand(hand: CanonicalHand | null): number[] | null {\\n  if (!hand) return Array<number>(featureSchema.handLength).fill(0);\\n  const points = hand.worldLandmarks;\\n  if (!points || !valid(points) || !valid(hand.landmarks) || hand.landmarks.some((point) => point.x < 0 || point.x > 1 || point.y < 0 || point.y > 1)) return null;\\n  // Array length was checked; coordinates here are anatomical landmark IDs, not hand order.\\n  const p = (index: number) => points[index]!;\\n  const scale = distance(p(0), p(9));\\n  if (scale < 1e-6) return null;\\n  const features: number[] = [];\\n  for (const [a, b, c] of joints) {\\n    const u = subtract(p(a), p(b));\\n    const v = subtract(p(c), p(b));\\n    const denominator = norm(u) * norm(v);\\n    if (denominator < 1e-12) return null;\\n    const cosine = (u.x * v.x + u.y * v.y + u.z * v.z) / denominator;\\n    features.push(Math.acos(Math.max(-1, Math.min(1, cosine))) / Math.PI);\\n  }\\n  for (const tip of tips.slice(1)) features.push(distance(p(4), p(tip)) / scale);\\n  for (const tip of tips) features.push(distance(p(0), p(tip)) / scale);\\n  for (const [a, b] of [[8, 12], [12, 16], [16, 20]]) features.push(distance(p(a!), p(b!)) / scale);\\n  const u = subtract(p(5), p(0));\\n  const v = subtract(p(17), p(0));\\n  const normal = { x: u.y * v.z - u.z * v.y, y: u.z * v.x - u.x * v.z, z: u.x * v.y - u.y * v.x };\\n  const normalLength = norm(normal);\\n  if (normalLength < 1e-12) return null;\\n  features.push(normal.x / normalLength, normal.y / normalLength, normal.z / normalLength);\\n  return features.length === featureSchema.handLength && features.every(Number.isFinite) ? features : null;\\n}\\n\\n/** Shared extraction for reference analysis and runtime. No hand mirroring or guessed data. */\\nexport function extractFeatures(frame: HandFrame): number[] | null {\\n  const left = extractHand(frame.left);\\n  const right = extractHand(frame.right);\\n  if (!left || !right || !Number.isFinite(frame.timestampMs)) return null;\\n  return [...left, ...right, Number(!!frame.left), Number(!!frame.right)];\\n}\\n\",\"scripts/load-training-contract.mjs\":\"import { readFile } from 'node:fs/promises';\\nimport ts from 'typescript';\\n\\n// Load the actual TypeScript contract/extractor for offline training. Imports are\\n// limited to these repository files; no alternate Python feature implementation.\\nexport async function loadTrainingContract() {\\n  const compile = async path => ts.transpileModule(await readFile(path,'utf8'), {compilerOptions:{module:ts.ModuleKind.ESNext,target:ts.ScriptTarget.ES2022}}).outputText;\\n  const url = text => `data:text/javascript;base64,${Buffer.from(text).toString('base64')}`;\\n  const configUrl = url(await compile('src/lib/config/tracking.ts'));\\n  const featureCode = (await compile('src/features/recognition/features.ts')).replaceAll('\\\"@/lib/config/tracking\\\"',JSON.stringify(configUrl));\\n  const featureModule = await import(url(featureCode));\\n  const contractModule = await import(url(await compile('ml/rhio/training-contract.ts')));\\n  return {...featureModule,...contractModule};\\n}\\r\\n\",\"scripts/colab/prepare-combined.mjs\":\"import { readFile, writeFile, mkdir, statfs } from 'node:fs/promises';\\nimport { createHash } from 'node:crypto';\\nconst hash = bytes => createHash('sha256').update(bytes).digest('hex');\\nconst root = '.tools/bisindo-dataset';\\nawait mkdir(root, { recursive: true });\\nlet stage = 'Menghubungi GitHub';\\nconst started = Date.now();\\nconsole.log('[Mulai] Memeriksa metadata kedua dataset...');\\nsetInterval(() => console.log(`[Status ${Math.round((Date.now()-started)/1000)}s] ${stage}`), 10000).unref();\\nconst get = async url => {\\n  for (let attempt = 0; attempt < 4; attempt++) {\\n    try { const r = await fetch(url, { signal: AbortSignal.timeout(url.includes('/public-files/') ? 120000 : 30000), headers: { Accept: new URL(url).hostname === 'api.github.com' ? 'application/vnd.github+json' : new URL(url).pathname.startsWith('/public-api/') ? 'application/vnd.mendeley-public-dataset.1+json' : '*/*', 'User-Agent': 'BISINDO-research-notebook' } }); if (r.ok) return r; throw new Error(`${r.status} ${url}: ${(await r.text()).slice(0, 400)}`); }\\n    catch (error) { console.warn(`[Retry ${attempt+1}/4] ${url}: ${error.message}`); if (attempt === 3) throw error; await new Promise(resolve => setTimeout(resolve, (attempt + 1) * 2000)); }\\n  }\\n};\\nconst repo = 'rhiosutoyo/Indonesian-Sign-Language-BISINDO-Hand-Sign-Detection-Dataset';\\nconst revision = 'e1a48c6caa9d12318c8561e9962da845ab50226e';\\nconst tree = await (await get(`https://api.github.com/repos/${repo}/git/trees/${revision}?recursive=1`)).json();\\nif (tree.truncated) throw new Error('GitHub tree truncated');\\nconst files = tree.tree.filter(f => /^(train|test)\\\\/[A-Z]\\\\.[^/]+\\\\.jpg$/.test(f.path));\\nconst samples = [];\\nlet metadataCursor = 0, metadataDone = 0;\\nawait Promise.all(Array.from({length: 4}, async () => {\\nwhile (metadataCursor < files.length) {\\n  const f = files[metadataCursor++];\\n  stage = `Metadata Rhio ${metadataDone}/${files.length}`;\\n  const xmlPath = f.path.replace(/\\\\.jpg$/, '.xml');\\n  const xml = await (await get(`https://raw.githubusercontent.com/${repo}/${revision}/${xmlPath}`)).text();\\n  const names = [...xml.matchAll(/<name>(.*?)<\\\\/name>/g)].map(m => m[1].trim());\\n  const label = f.path.split('/')[1][0];\\n  if (names.length !== 1 || names[0] !== label) throw new Error('XML label conflict: ' + f.path);\\n  samples.push({ id: 'rhio/' + f.path, label, sourceId: 'rhio-bisindo-2024', sourceUrl: `https://raw.githubusercontent.com/${repo}/${revision}/${f.path}`, gitBlobSha: f.sha, bytes: f.size, publisherSplit: f.path.startsWith('test/') ? 'test' : 'train', license: 'MIT', signerId: null });\\n  metadataDone++;\\n  if (metadataDone % 20 === 0 || metadataDone === files.length) console.log(`[Rhio] XML diperiksa ${metadataDone}/${files.length}`);\\n}\\n}));\\nstage = 'Mengambil struktur folder Mendeley';\\nconsole.log('[Mendeley] Membaca folder Original Images');\\nconst base = 'https://data.mendeley.com/public-api/datasets/ywnjpbcz8m';\\nconst folders = await (await get(base + '/folders/1')).json();\\nconst originals = folders.find(f => f.name === '01. Original Images');\\nif (!originals) throw new Error('Original image directory missing; do not substitute resized/binary derivatives');\\nfor (const folder of folders.filter(f => f.parent_id === originals.id && /^[A-Z]$/.test(f.name))) {\\n  stage = `Metadata Mendeley huruf ${folder.name}`;\\n  console.log('[Mendeley] Mengambil daftar huruf', folder.name);\\n  const list = await (await get(`${base}/files?folder_id=${folder.id}&version=1`)).json();\\n  if (!Array.isArray(list)) throw new Error('Unexpected Mendeley file response');\\n  for (const f of list) {\\n    if (!/^image\\\\//.test(f.content_details?.content_type ?? '')) continue;\\n    samples.push({ id: 'sanjaya/' + f.id + '.jpg', label: folder.name, sourceId: 'sanjaya-bisindo-alphabet-2024-v1', sourceUrl: f.content_details.download_url, sha256: f.content_details.sha256_hash, bytes: f.size, originalFilename: f.filename, publisherSplit: null, license: 'CC BY 4.0', signerId: null });\\n  }\\n}\\nfor (const source of ['rhio-bisindo-2024', 'sanjaya-bisindo-alphabet-2024-v1']) for (const letter of 'ABCDEFGHIJKLMNOPQRSTUVWXYZ') {\\n  if (!samples.some(s => s.sourceId === source && s.label === letter)) throw new Error(`Missing source label ${source}/${letter}`);\\n}\\nconsole.log('Originals:', samples.length, 'Download GB:', (samples.reduce((s, r) => s + r.bytes, 0) / 1e9).toFixed(2));\\nconst disk = await statfs('.');\\nif (disk.bavail * disk.bsize < samples.reduce((s, r) => s + r.bytes, 0) + 2e9) throw new Error('Insufficient disk for original datasets; use a larger runtime/disk');\\nstage = 'Mulai unduh / verifikasi foto';\\nlet cursor = 0, done = 0;\\nawait Promise.all(Array.from({ length: 4 }, async () => {\\n  while (cursor < samples.length) {\\n    const sample = samples[cursor++], path = root + '/' + sample.id;\\n    await mkdir(path.slice(0, path.lastIndexOf('/')), { recursive: true });\\n    let bytes; try { bytes = await readFile(path); } catch {}\\n    const valid = b => b && (sample.sha256 ? hash(b) === sample.sha256 : createHash('sha1').update(`blob ${b.length}\\\\0`).update(b).digest('hex') === sample.gitBlobSha);\\n    if (!valid(bytes)) { bytes = Buffer.from(await (await get(sample.sourceUrl)).arrayBuffer()); if (!valid(bytes)) throw new Error('Checksum mismatch: ' + sample.id); await writeFile(path, bytes); }\\n    sample.sha256 = hash(bytes); sample.groupId = sample.sha256;\\n    done++; stage = `Foto diverifikasi ${done}/${samples.length}`;\\n    if (done % 20 === 0 || done === samples.length) console.log('[Foto]', done, '/', samples.length);\\n  }\\n}));\\nstage = 'Menyusun split dan manifest';\\n// Exact duplicates never cross splits. Conflicting duplicate labels stop training.\\nconst duplicateGroups = new Map();\\nfor (const sample of samples) {\\n  const group = duplicateGroups.get(sample.sha256) ?? [];\\n  group.push(sample);\\n  duplicateGroups.set(sample.sha256, group);\\n}\\nconst unique = new Map();\\nconst conflicts = [];\\nfor (const [sha256, group] of duplicateGroups) {\\n  if (new Set(group.map(sample => sample.label)).size > 1) {\\n    conflicts.push({ sha256, samples: group.map(({ id, label, sourceId }) => ({ id, label, sourceId })) });\\n    continue;\\n  }\\n  unique.set(sha256, group.find(sample => sample.publisherSplit === 'test') ?? group[0]);\\n}\\nawait mkdir('output', { recursive: true });\\nawait writeFile('output/label-conflicts.json', JSON.stringify(conflicts, null, 2));\\nconsole.log('Grup foto konflik dikeluarkan:', conflicts.length);\\nconst selected = [...unique.values()];\\nfor (const source of new Set(selected.map(s => s.sourceId))) for (const label of 'ABCDEFGHIJKLMNOPQRSTUVWXYZ') {\\n  const group = selected.filter(s => s.sourceId === source && s.label === label).sort((a, b) => a.sha256.localeCompare(b.sha256));\\n  if (source === 'rhio-bisindo-2024') {\\n    const training = group.filter(s => s.publisherSplit !== 'test');\\n    for (const s of group) s.split = s.publisherSplit === 'test' ? 'test' : training.indexOf(s) < Math.max(1, Math.floor(training.length * .2)) ? 'validation' : 'train';\\n  } else {\\n    const n = Math.max(1, Math.floor(group.length * .2));\\n    group.forEach((s, i) => { s.split = i < n ? 'test' : i < 2 * n ? 'validation' : 'train'; });\\n  }\\n}\\nawait mkdir('ml/rhio', { recursive: true });\\nconst sources = [{ id: 'rhio-bisindo-2024', repo, revision, license: 'MIT' }, { id: 'sanjaya-bisindo-alphabet-2024-v1', doi: '10.17632/ywnjpbcz8m.1', author: 'Samuel Ady Sanjaya', license: 'CC BY 4.0', url: 'https://data.mendeley.com/datasets/ywnjpbcz8m/1' }];\\nawait writeFile('ml/rhio/manifest.json', JSON.stringify({ sources, splitLimitation: 'Original-image groups only; signer/session metadata unknown. No augmentation or derivative image versions.', samples: selected }, null, 2));\\nawait mkdir('output', { recursive: true });\\nawait writeFile('output/LICENSE.rhio', await (await get(`https://raw.githubusercontent.com/${repo}/${revision}/LICENSE`)).text());\\nawait writeFile('output/ATTRIBUTION.txt', 'Combined source-labelled static-photo experiment.\\\\nRhio Sutoyo et al., BISINDO Hand-Sign Detection Dataset, MIT. See LICENSE.rhio.\\\\nSamuel Ady Sanjaya (2024), BISINDO Indonesian Sign Language: Alphabet Image Data, v1. DOI 10.17632/ywnjpbcz8m.1. CC BY 4.0 https://creativecommons.org/licenses/by/4.0/ . Original images converted to landmark features; model fitted from features.\\\\nNo claim of mentor validation or live recognition accuracy.\\\\n');\\nconsole.log('Manifest ready:', selected.length, 'original photos; duplicate copies removed:', samples.length - selected.length);\\n\",\"public/models/mediapipe/provenance.json\":\"{\\r\\n  \\\"package\\\": \\\"@mediapipe/tasks-vision\\\",\\r\\n  \\\"packageVersion\\\": \\\"1.0.1\\\",\\r\\n  \\\"license\\\": \\\"Apache-2.0\\\",\\r\\n  \\\"modelSource\\\": \\\"https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task\\\",\\r\\n  \\\"modelCard\\\": \\\"https://storage.googleapis.com/mediapipe-assets/Model%20Card%20Hand%20Tracking%20(Lite_Full)%20with%20Fairness%20Oct%202021.pdf\\\",\\r\\n  \\\"licenseSource\\\": \\\"https://github.com/google-ai-edge/mediapipe/blob/master/LICENSE\\\",\\r\\n  \\\"fetchedAt\\\": \\\"2026-09-11\\\",\\r\\n  \\\"assets\\\": [\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/hand_landmarker-float16-v1.task\\\",\\r\\n      \\\"bytes\\\": 7819105,\\r\\n      \\\"sha256\\\": \\\"fbc2a30080c3c557093b5ddfc334698132eb341044ccee322ccf8bcf3607cde1\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_internal.js\\\",\\r\\n      \\\"bytes\\\": 323377,\\r\\n      \\\"sha256\\\": \\\"e170ee67dd4e16c1a6fcd8840a206687e5a59b22c20e4a902bc445b095454d73\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_internal.wasm\\\",\\r\\n      \\\"bytes\\\": 11756954,\\r\\n      \\\"sha256\\\": \\\"8da277a733926eacd0474b8704b36742d6ec3231c57a860c5b889dff8f1df886\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_module_internal.js\\\",\\r\\n      \\\"bytes\\\": 323415,\\r\\n      \\\"sha256\\\": \\\"da8934057f147b622e82cfb4c0dbd85461c598e268588b5a8ba9ca963a8ff82d\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_module_internal.wasm\\\",\\r\\n      \\\"bytes\\\": 11756972,\\r\\n      \\\"sha256\\\": \\\"2dabd8e23c60984628beb7bb338764c81a08e6837145273f59578684b5d53c1b\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_nosimd_internal.js\\\",\\r\\n      \\\"bytes\\\": 323180,\\r\\n      \\\"sha256\\\": \\\"e81d715a3d42cc3373602eb2f7aff795d164934db680e32496b65dab537f9658\\\"\\r\\n    },\\r\\n    {\\r\\n      \\\"path\\\": \\\"public/models/mediapipe/tasks-vision-1.0.1/wasm/vision_wasm_nosimd_internal.wasm\\\",\\r\\n      \\\"bytes\\\": 10960242,\\r\\n      \\\"sha256\\\": \\\"a28483cd42e74e855bf5ebdb6b40d9b66a5b49e35e95020bc97669e6822a3192\\\"\\r\\n    }\\r\\n  ]\\r\\n}\\r\\n\",\"ml/rhio/training-contract.ts\":\"import type { HandFrame } from \\\"../../src/types/tracking\\\";\\n\\ntype SourceSample = { id: string; label: string; sha256: string; groupId: string; split: string; publisherSplit: string };\\ntype ExtractionRow = SourceSample & { tick: number; frame: HandFrame; vector: number[] | null };\\ntype Schema = { id: string; length: number; normalizationVersion: string; landmarkerAssetId: string };\\n\\n/** Fail before fitting when a cached extraction no longer agrees with its source manifest/runtime. */\\nexport function validateTrainingData(\\n  manifest: { samples: SourceSample[] },\\n  dataset: { featureSchema: Schema; rows: ExtractionRow[] },\\n  expectedSchema: Schema,\\n  extract: (frame: HandFrame) => number[] | null,\\n): void {\\n  for (const key of [\\\"id\\\", \\\"length\\\", \\\"normalizationVersion\\\", \\\"landmarkerAssetId\\\"] as const) {\\n    if (dataset.featureSchema[key] !== expectedSchema[key]) throw new Error(`Feature schema mismatch: ${key}`);\\n  }\\n  const sources = new Map<string, SourceSample>();\\n  const groups = new Map<string, string>();\\n  for (const sample of manifest.samples) {\\n    if (sources.has(sample.id) || !/^[A-Z]$/.test(sample.label) || ![\\\"train\\\", \\\"validation\\\", \\\"test\\\"].includes(sample.split) || !/^[a-f0-9]{64}$/.test(sample.sha256) || !sample.groupId) throw new Error(\\\"Invalid source manifest\\\");\\n    if (sample.publisherSplit !== null && (sample.publisherSplit === \\\"test\\\") !== (sample.split === \\\"test\\\")) throw new Error(\\\"Publisher test partition changed\\\");\\n    for (const group of [sample.groupId, sample.sha256]) {\\n      if (groups.has(group) && groups.get(group) !== sample.split) throw new Error(\\\"Source group crosses partitions\\\");\\n      groups.set(group, sample.split);\\n    }\\n    sources.set(sample.id, sample);\\n  }\\n  const observations = new Set<string>();\\n  for (const row of dataset.rows) {\\n    const source = sources.get(row.id);\\n    if (!source || [\\\"label\\\", \\\"sha256\\\", \\\"groupId\\\", \\\"split\\\", \\\"publisherSplit\\\"].some(key => row[key as keyof SourceSample] !== source[key as keyof SourceSample])) throw new Error(`Extraction provenance mismatch: ${row.id}`);\\n    const observation = `${row.id}:${row.tick}`;\\n    if (!Number.isInteger(row.tick) || row.tick < 0 || observations.has(observation)) throw new Error(\\\"Duplicate or invalid extraction tick\\\");\\n    observations.add(observation);\\n    if (row.vector === null) continue;\\n    const actual = extract(row.frame);\\n    if (!actual || row.vector.length !== expectedSchema.length || !row.vector.every(Number.isFinite) || actual.some((v,i) => Math.abs(v-row.vector![i]!)>1e-10)) throw new Error(`Feature parity mismatch: ${row.id}`);\\n  }\\n  if (!observations.size) throw new Error(\\\"No extraction observations\\\");\\n}\\r\\n\",\"scripts/extract-alphabet-landmarks.mjs\":\"import { chromium } from '@playwright/test';\\nimport { readFile, writeFile, appendFile, rename } from 'node:fs/promises';\\nimport ts from 'typescript';\\n\\nconst manifest = JSON.parse(await readFile('ml/rhio/manifest.json', 'utf8'));\\nconst checkpointPath = '.tools/bisindo-dataset/features-checkpoint.ndjson';\\nconst finalPath = '.tools/bisindo-dataset/features.json';\\nconst sampleById = new Map(manifest.samples.map(sample => [sample.id, sample]));\\nconst completed = new Map();\\n\\ntry {\\n  const lines = (await readFile(checkpointPath, 'utf8')).split('\\\\n');\\n  for (let index = 0; index < lines.length; index++) {\\n    const line = lines[index].trim();\\n    if (!line) continue;\\n    let row;\\n    try { row = JSON.parse(line); }\\n    catch (error) {\\n      if (index === lines.length - 1) { console.warn('Checkpoint terakhir terpotong; baris itu diabaikan.'); continue; }\\n      throw new Error('Checkpoint rusak pada baris ' + (index + 1), { cause: error });\\n    }\\n    const sample = sampleById.get(row.id);\\n    if (!sample || row.label !== sample.label || row.sha256 !== sample.sha256 || row.groupId !== sample.groupId || row.split !== sample.split || row.sourceId !== sample.sourceId) {\\n      throw new Error('Checkpoint tidak cocok dengan manifest: ' + row.id);\\n    }\\n    completed.set(row.id, row);\\n  }\\n} catch (error) {\\n  if (error.code !== 'ENOENT') throw error;\\n}\\n\\nconst pending = manifest.samples.filter(sample => !completed.has(sample.id));\\nconsole.log('Checkpoint:', completed.size, 'selesai; tersisa:', pending.length);\\n\\nconst modules = {\\n  '/__train/vision.mjs': 'node_modules/@mediapipe/tasks-vision/vision_bundle.mjs',\\n  '/__train/config.mjs': 'src/lib/config/tracking.ts',\\n  '/__train/canonicalize.mjs': 'src/lib/mediapipe/canonicalize.ts',\\n  '/__train/features.mjs': 'src/features/recognition/features.ts',\\n};\\nconst browser = await chromium.launch({ args: ['--no-sandbox'] });\\nlet saveQueue = Promise.resolve();\\nlet savedThisRun = 0;\\nlet schema;\\nlet runtimeConfig;\\n\\nasync function configurePage(page) {\\n  await page.route('**/__train/*.mjs', async route => {\\n    const file = modules[new URL(route.request().url()).pathname];\\n    if (!file) return route.abort();\\n    let body = await readFile(file, 'utf8');\\n    if (file.endsWith('.ts')) body = ts.transpileModule(body, { compilerOptions: { module: ts.ModuleKind.ESNext, target: ts.ScriptTarget.ES2022 } }).outputText.replaceAll('\\\"@/lib/config/tracking\\\"', '\\\"/__train/config.mjs\\\"');\\n    await route.fulfill({ contentType: 'text/javascript', body });\\n  });\\n  await page.route('**/__dataset/**', async route => {\\n    const id = decodeURIComponent(new URL(route.request().url()).pathname.slice('/__dataset/'.length));\\n    if (!sampleById.has(id)) return route.abort();\\n    await route.fulfill({ contentType: 'image/jpeg', body: await readFile('.tools/bisindo-dataset/' + id) });\\n  });\\n  await page.route('**/models/**', async route => {\\n    const path = new URL(route.request().url()).pathname;\\n    if (path.includes('..')) return route.abort();\\n    const contentType = path.endsWith('.js') ? 'text/javascript' : path.endsWith('.wasm') ? 'application/wasm' : 'application/octet-stream';\\n    await route.fulfill({ contentType, body: await readFile('public' + path) });\\n  });\\n  await page.route('http://127.0.0.1:3000/credits', route => route.fulfill({ contentType: 'text/html', headers: { 'Content-Security-Policy': \\\"connect-src 'self'\\\" }, body: '<!doctype html><title>Offline extraction</title>' }));\\n  await page.goto('http://127.0.0.1:3000/credits');\\n  await page.exposeFunction('saveTrainingRow', row => {\\n    saveQueue = saveQueue.then(async () => {\\n      if (completed.has(row.id)) return;\\n      await appendFile(checkpointPath, JSON.stringify(row) + '\\\\n');\\n      completed.set(row.id, row);\\n      savedThisRun++;\\n      if (savedThisRun % 10 === 0 || completed.size === manifest.samples.length) {\\n        console.log('Saved', completed.size + '/' + manifest.samples.length, '(run ini ' + savedThisRun + '/' + pending.length + ')');\\n      }\\n    });\\n    return saveQueue;\\n  });\\n}\\n\\nasync function runWorker(samples, workerIndex) {\\n  const page = await browser.newPage();\\n  await configurePage(page);\\n  try {\\n    return await page.evaluate(async ({ samples, workerIndex }) => {\\n      const { FilesetResolver, HandLandmarker } = await import('/__train/vision.mjs');\\n      const { trackingConfig } = await import('/__train/config.mjs');\\n      const { canonicalizeHands } = await import('/__train/canonicalize.mjs');\\n      const { featureSchema, extractFeatures } = await import('/__train/features.mjs');\\n      const fileset = await FilesetResolver.forVisionTasks(trackingConfig.wasmRoot);\\n      const model = await HandLandmarker.createFromOptions(fileset, {\\n        baseOptions: { modelAssetPath: trackingConfig.modelPath, delegate: 'CPU' },\\n        runningMode: 'IMAGE', numHands: 2,\\n        minHandDetectionConfidence: .5, minHandPresenceConfidence: .5, minTrackingConfidence: .5,\\n      });\\n      try {\\n        for (let index = 0; index < samples.length; index++) {\\n          const sample = samples[index];\\n          const timestampMs = workerIndex * 100000000 + index;\\n          const emptyFrame = { timestampMs, left: null, right: null };\\n          const image = new Image();\\n          image.src = '/__dataset/' + encodeURIComponent(sample.id);\\n          try { await image.decode(); }\\n          catch {\\n            await window.saveTrainingRow({ ...sample, tick: 0, vector: null, frame: emptyFrame, rejection: 'IMAGE_DECODE' });\\n            continue;\\n          }\\n          const canvas = document.createElement('canvas');\\n          canvas.width = 640; canvas.height = 480;\\n          const context = canvas.getContext('2d');\\n          context.fillStyle = '#ffffff'; context.fillRect(0, 0, 640, 480);\\n          const scale = Math.min(640 / image.naturalWidth, 480 / image.naturalHeight);\\n          const width = image.naturalWidth * scale, height = image.naturalHeight * scale;\\n          context.drawImage(image, (640 - width) / 2, (480 - height) / 2, width, height);\\n          const result = canonicalizeHands(model.detect(canvas), timestampMs);\\n          const count = Number(!!result.frame.left) + Number(!!result.frame.right);\\n          const vector = !result.ambiguous && count >= 1 ? extractFeatures(result.frame) : null;\\n          await window.saveTrainingRow({ ...sample, tick: 0, vector, frame: result.frame, rejection: result.ambiguous ? 'AMBIGUOUS' : count < 1 ? 'NO_HAND' : vector ? null : 'INVALID_FEATURES' });\\n        }\\n      } finally { model.close(); }\\n      return { featureSchema, trackingConfig };\\n    }, { samples, workerIndex });\\n  } finally { await page.close(); }\\n}\\n\\ntry {\\n  const workerCount = Math.min(2, Math.max(1, pending.length));\\n  const groups = Array.from({ length: workerCount }, (_, worker) => pending.filter((_, index) => index % workerCount === worker));\\n  const results = await Promise.all(groups.map((samples, index) => runWorker(samples, index)));\\n  schema = results[0]?.featureSchema;\\n  runtimeConfig = results[0]?.trackingConfig;\\n  await saveQueue;\\n} finally { await browser.close(); }\\n\\nif (completed.size !== manifest.samples.length) throw new Error('Checkpoint belum lengkap: ' + completed.size + '/' + manifest.samples.length);\\nconst rows = manifest.samples.map(sample => completed.get(sample.id));\\nconst temporary = finalPath + '.tmp';\\nawait writeFile(temporary, JSON.stringify({ featureSchema: schema, trackingConfig: runtimeConfig, rows }));\\nawait rename(temporary, finalPath);\\nconst counts = rows.reduce((result, row) => { const key = row.rejection ?? 'USABLE'; result[key] = (result[key] ?? 0) + 1; return result; }, {});\\nconsole.log('SELESAI', rows.length, 'foto:', counts);\\n\",\"scripts/colab/train-alphabet-cpu.mjs\":\"import { readFile, writeFile, mkdir } from 'node:fs/promises';\\nimport { createHash } from 'node:crypto';\\nimport { RandomForestClassifier } from 'ml-random-forest';\\nimport { loadTrainingContract } from '../load-training-contract.mjs';\\nconst read = async path => JSON.parse(await readFile(path, 'utf8'));\\nconst dataset = await read('.tools/bisindo-dataset/features.json');\\nconst manifest = await read('ml/rhio/manifest.json');\\nconst { validateTrainingData, featureSchema, extractFeatures } = await loadTrainingContract();\\nvalidateTrainingData(manifest, dataset, featureSchema, extractFeatures);\\nconst labels = [...'ABCDEFGHIJKLMNOPQRSTUVWXYZ'];\\nconst usable = dataset.rows.filter(row => row.vector);\\nconst partitions = Object.fromEntries(['train', 'validation', 'test'].map(split => [split, usable.filter(row => row.split === split)]));\\nconst coverage = labels.map(label => ({ label, ...Object.fromEntries(Object.entries(partitions).map(([split, rows]) => [split, rows.filter(row => row.label === label).length])) }));\\nawait mkdir('output', { recursive: true });\\nawait writeFile('output/coverage.json', JSON.stringify(coverage, null, 2));\\nconst missing = coverage.filter(row => row.train < 2 || row.validation < 1 || row.test < 1);\\nif (missing.length) throw new Error('Data ekstraksi belum cukup; lihat output/coverage.json: ' + missing.map(row => row.label).join(', '));\\nconst options = { seed: 42, nEstimators: 160, maxFeatures: .65, replacement: false, useSampleBagging: true, noOOB: true, treeOptions: { maxDepth: 16, minNumSamples: 2 } };\\nconsole.log('Training Random Forest CPU:', partitions.train.length, 'train rows');\\nconst started = Date.now();\\nconst forest = new RandomForestClassifier(options);\\nforest.train(partitions.train.map(row => row.vector), partitions.train.map(row => labels.indexOf(row.label)));\\nconst votes = vector => {\\n  const counts = labels.map(() => 0);\\n  for (const prediction of forest.predictionValues([vector]).getRow(0)) counts[prediction]++;\\n  return counts.map(count => count / options.nEstimators);\\n};\\nconst ranked = vector => votes(vector).map((score, index) => ({ score, index })).sort((a, b) => b.score - a.score);\\nconst distance = (left, right) => Math.sqrt(left.reduce((sum, value, index) => sum + (value - right[index]) ** 2, 0) / left.length);\\nconst sameMask = (left, right) => left[50] === right[50] && left[51] === right[51];\\nconst envelopes = labels.map(label => {\\n  const positives = partitions.train.filter(row => row.label === label);\\n  const distances = [...positives, ...partitions.validation.filter(row => row.label === label)]\\n    .map(row => Math.min(...positives.filter(candidate => candidate.groupId !== row.groupId && sameMask(candidate.vector, row.vector)).map(candidate => distance(candidate.vector, row.vector))))\\n    .filter(Number.isFinite).sort((a, b) => a - b);\\n  if (!distances.length) throw new Error('Tidak ada distance coverage: ' + label);\\n  return { label, maxDistance: distances[Math.ceil(distances.length * .95) - 1], vectors: positives.map(row => row.vector) };\\n});\\nconst classify = (vector, threshold) => {\\n  const ranking = ranked(vector), best = ranking[0], envelope = envelopes[best.index];\\n  const nearest = Math.min(...envelope.vectors.filter(candidate => sameMask(candidate, vector)).map(candidate => distance(candidate, vector)));\\n  return best.score >= threshold && best.score > ranking[1].score && nearest <= envelope.maxDistance ? labels[best.index] : null;\\n};\\nconst calibrations = [.5, .55, .6, .65, .7, .75, .8, .85, .9, .95, 1].map(threshold => {\\n  let correct = 0, wrong = 0;\\n  for (const row of partitions.validation) { const prediction = classify(row.vector, threshold); if (prediction === row.label) correct++; else if (prediction !== null) wrong++; }\\n  return { threshold, correct, wrong, uncertain: partitions.validation.length - correct - wrong, utility: correct - 4 * wrong };\\n}).sort((a, b) => b.utility - a.utility || a.wrong - b.wrong || b.threshold - a.threshold);\\nconst calibration = calibrations[0];\\nconst confusion = labels.map(() => Array(labels.length + 1).fill(0));\\nfor (const row of partitions.test) { const prediction = classify(row.vector, calibration.threshold); confusion[labels.indexOf(row.label)][prediction === null ? labels.length : labels.indexOf(prediction)]++; }\\nconst payload = { id: 'combined-alphabet-rf-candidate-v1', version: '1.0.0', status: 'EXPERIMENTAL', deploymentReady: false, labels, featureSchema: dataset.featureSchema, runtime: { name: 'ml-random-forest', version: '2.1.0' }, landmarker: dataset.trackingConfig, sources: manifest.sources, threshold: calibration.threshold, envelopes, forest: forest.toJSON() };\\nconst serialized = JSON.stringify(payload);\\nawait writeFile('output/model.json', serialized);\\nconst loaded = RandomForestClassifier.load(JSON.parse(serialized).forest);\\nif (JSON.stringify(loaded.predict(partitions.test.map(row => row.vector))) !== JSON.stringify(forest.predict(partitions.test.map(row => row.vector)))) throw new Error('Export parity gagal');\\nconst sourceById = new Map(manifest.samples.map(sample => [sample.id, sample.sourceId]));\\nconst report = {\\n  modelSha256: createHash('sha256').update(serialized).digest('hex'), options,\\n  training: { device: 'CPU', seconds: (Date.now() - started) / 1000 }, coverage, calibration,\\n  extraction: { total: dataset.rows.length, usable: usable.length, rejected: dataset.rows.length - usable.length },\\n  testConfusion: { rows: labels, columns: [...labels, 'UNCERTAIN'], values: confusion },\\n  perLetter: labels.map((label, index) => ({ label, tested: confusion[index].reduce((a, b) => a + b, 0), matched: confusion[index][index], uncertain: confusion[index][labels.length], wrong: confusion[index].reduce((a, b, column) => a + (column !== index && column !== labels.length ? b : 0), 0) })),\\n  perSource: [...new Set(manifest.samples.map(sample => sample.sourceId))].flatMap(sourceId => labels.map(label => {\\n    const rows = partitions.test.filter(row => sourceById.get(row.id) === sourceId && row.label === label);\\n    const predictions = rows.map(row => classify(row.vector, calibration.threshold));\\n    return { sourceId, label, tested: rows.length, matched: predictions.filter(value => value === label).length, uncertain: predictions.filter(value => value === null).length, wrong: predictions.filter(value => value !== null && value !== label).length };\\n  })),\\n  limitations: ['Static photo-label experiment, not validated dynamic alphabet recognition.', 'Signer/session identities unavailable; split is not signer-independent.', 'No unknown-pose or live webcam acceptance evaluation.', 'Website integration and browser acceptance tests are still required.'],\\n  golden: labels.flatMap(label => partitions.test.filter(row => row.label === label).slice(0, 2)).map(row => ({ id: row.id, label: row.label, frame: row.frame, vector: row.vector, votes: votes(row.vector), predicted: classify(row.vector, calibration.threshold) })),\\n};\\nawait writeFile('output/evaluation.json', JSON.stringify(report, null, 2));\\nconsole.log(JSON.stringify({ ...report, golden: undefined, testConfusion: undefined }, null, 2));\\n\"}")
for name, content in bundle.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
package = {'private': True, 'type': 'module', 'dependencies': {'@mediapipe/tasks-vision':'1.0.1', '@playwright/test':'1.63.0', 'typescript':'5.9.3', 'ml-random-forest':'2.1.0'}}
(ROOT / 'package.json').write_text(json.dumps(package), encoding='utf-8')
run_live('npm', 'install', '--no-audit', '--no-fund')
run_live('npx', 'playwright', 'install', '--with-deps', 'chromium')
provenance = json.loads((ROOT / 'public/models/mediapipe/provenance.json').read_text())
for asset in provenance['assets']:
    target = ROOT / asset['path']
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.suffix == '.task':
        if not target.exists(): urllib.request.urlretrieve(provenance['modelSource'], target)
    else:
        shutil.copyfile(ROOT / 'node_modules/@mediapipe/tasks-vision/wasm' / target.name, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == asset['sha256'], f'Checksum gagal: {target}'
print('Setup selesai: MediaPipe dan seluruh source tervalidasi.')


## 1. Unduh dan siapkan dataset

Sel ini bisa lama karena total foto asli sekitar 8.5 GB. File dengan checksum yang sudah benar tidak diunduh ulang. Konflik label dan duplikat ditulis ke folder output.

In [ ]:
run_live('node', 'scripts/colab/prepare-combined.mjs')
manifest = json.loads((ROOT / 'ml/rhio/manifest.json').read_text())
from collections import Counter
print(Counter((row['sourceId'], row['split']) for row in manifest['samples']))


## 2. Tinjau contoh kedua sumber

Ubah LETTER untuk melihat label lain. Ini pemeriksaan visual, bukan validasi linguistik semua foto.

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
LETTER = 'C'
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for row, source in enumerate(['rhio-bisindo-2024', 'sanjaya-bisindo-alphabet-2024-v1']):
    samples = [item for item in manifest['samples'] if item['label'] == LETTER and item['sourceId'] == source][:3]
    for ax, sample in zip(axes[row], samples):
        with Image.open(ROOT / '.tools/bisindo-dataset' / sample['id']) as image:
            ax.imshow(ImageOps.exif_transpose(image))
        ax.set_title(source.split('-')[0] + ' / ' + LETTER); ax.axis('off')
plt.tight_layout(); plt.show()


## 3. Ekstrak landmark - CPU, dua worker, resume otomatis

Setiap foto langsung ditambahkan ke checkpoint. Gambar yang gagal didecode dicatat sebagai IMAGE_DECODE dan proses berlanjut. Error sistem menghentikan proses agar tidak menyamarkan kerusakan; jalankan ulang sel ini untuk melanjutkan dari checkpoint.

In [ ]:
run_live('node', 'scripts/extract-alphabet-landmarks.mjs')
features = json.loads((ROOT / '.tools/bisindo-dataset/features.json').read_text())
print(Counter(row['rejection'] or 'USABLE' for row in features['rows']))


## 4. Training Random Forest - CPU

Training menggunakan split yang sudah dibuat, threshold dikalibrasi pada validation, lalu test diperiksa sekali. Kelas yang tidak memiliki coverage minimum akan menghentikan training dengan laporan yang jelas.

In [ ]:
run_live('node', 'scripts/colab/train-alphabet-cpu.mjs')
report = json.loads((ROOT / 'output/evaluation.json').read_text())
import pandas as pd
display(pd.DataFrame(report['perLetter']))
display(pd.DataFrame(report['perSource']))


In [ ]:
import numpy as np
matrix = np.array(report['testConfusion']['values'])
fig, ax = plt.subplots(figsize=(15, 12))
chart = ax.imshow(matrix, cmap='Blues')
ax.set_xticks(range(len(report['testConfusion']['columns'])), report['testConfusion']['columns'], rotation=90)
ax.set_yticks(range(len(report['testConfusion']['rows'])), report['testConfusion']['rows'])
ax.set_xlabel('Prediksi'); ax.set_ylabel('Label sumber'); fig.colorbar(chart)
plt.tight_layout(); plt.savefig(ROOT / 'output/confusion.png'); plt.show()


## 5. Download hasil

ZIP tidak menyertakan foto dataset. Simpan model, evaluasi, manifest, konflik label, provenance dan atribusi. Jangan menimpa model C/L/O website sebelum integrasi dan browser tests selesai.

In [ ]:
shutil.copyfile(ROOT / 'ml/rhio/manifest.json', ROOT / 'output/manifest.json')
shutil.copyfile(ROOT / 'public/models/mediapipe/provenance.json', ROOT / 'output/mediapipe-provenance.json')
shutil.copyfile(ROOT / 'package-lock.json', ROOT / 'output/package-lock.json')
shutil.make_archive('/content/bisindo-alphabet-results', 'zip', ROOT / 'output')
from google.colab import files
files.download('/content/bisindo-alphabet-results.zip')
